#환경설정

In [ ]:
# facemap_3dmm 포함 qai_hub_models 설치
!pip install -q 'qai-hub-models[facemap-3dmm]'

# InsightFace + ONNX Runtime GPU
!pip install -q insightface onnxruntime-gpu opencv-python-headless

print('✅ 설치 완료')

In [ ]:
import os

os.makedirs('/root/.insightface/models', exist_ok=True)
INSWAPPER_PATH = '/root/.insightface/models/inswapper_128.onnx'

if not os.path.exists(INSWAPPER_PATH):
    print('⬇ inswapper_128.onnx 다운로드 중... (~500MB)')
    !wget -q --show-progress \
        -O {INSWAPPER_PATH} \
        https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx
    print('✅ 다운로드 완료')
else:
    print('✅ inswapper_128.onnx 이미 존재')

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── facemap_3dmm 로드 ────────────────────────────────────────────────────────
from qai_hub_models.models.facemap_3dmm import Model as FaceMap3DMM

print('facemap_3dmm 로드 중...')
facemap_model = FaceMap3DMM.from_pretrained()
facemap_model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
facemap_model = facemap_model.to(device)

print(f'✅ facemap_3dmm 로드 완료 | device: {device}')
print(f'   입력 spec: {facemap_model.get_input_spec()}')

# 모델 초기화 및 전처리


In [ ]:
import insightface
from insightface.app import FaceAnalysis

# FaceAnalysis: 얼굴 감지 + 임베딩
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0, det_size=(640, 640))

# inswapper 로드
swapper = insightface.model_zoo.get_model(
    INSWAPPER_PATH, download=False
)

print(f'✅ InsightFace 초기화 완료')

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── facemap_3dmm 로드 ────────────────────────────────────────────────────────
from qai_hub_models.models.facemap_3dmm import Model as FaceMap3DMM

print('facemap_3dmm 로드 중...')
facemap_model = FaceMap3DMM.from_pretrained()
facemap_model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
facemap_model = facemap_model.to(device)

print(f'✅ facemap_3dmm 로드 완료 | device: {device}')
print(f'   입력 spec: {facemap_model.get_input_spec()}')

# ──────────────────────────────────────────────────────────────────────────
# facemap_3dmm 입출력 처리
#
# 입력: 128x128 RGB float32 텐서 (정규화: mean=0.5, std=0.5)
# 출력: 랜드마크 좌표 텐서 (128x128 좌표 공간)
#       → 실제 이미지 크기로 역변환 필요
#
# 5-point keypoints 매핑 (iBUG 68점 기준):
#   오른눈: 36~41, 왼눈: 42~47, 코끝: 30, 우입꼬리: 48, 좌입꼬리: 54
# ──────────────────────────────────────────────────────────────────────────

INPUT_SIZE = 128  # facemap_3dmm 고정 입력 크기

def preprocess_for_facemap(img_bgr: np.ndarray, face_bbox=None) -> tuple:
    """
    이미지 → facemap_3dmm 입력 텐서

    Args:
        img_bgr  : 원본 BGR 이미지
        face_bbox: [x1,y1,x2,y2] 얼굴 박스 (없으면 전체 이미지 사용)
    Returns:
        tensor   : (1, 3, 128, 128) float32 CUDA/CPU 텐서
        scale_xy : (scale_x, scale_y) — 랜드마크 역변환용
        offset_xy: (offset_x, offset_y) — 랜드마크 역변환용
    """
    h, w = img_bgr.shape[:2]

    if face_bbox is not None:
        x1, y1, x2, y2 = [int(v) for v in face_bbox]
        # 박스 패딩 (20%)
        pad_x = int((x2 - x1) * 0.2)
        pad_y = int((y2 - y1) * 0.2)
        x1 = max(0, x1 - pad_x)
        y1 = max(0, y1 - pad_y)
        x2 = min(w, x2 + pad_x)
        y2 = min(h, y2 + pad_y)
        crop = img_bgr[y1:y2, x1:x2]
        offset_xy = (x1, y1)
        crop_wh   = (x2 - x1, y2 - y1)
    else:
        crop = img_bgr
        offset_xy = (0, 0)
        crop_wh   = (w, h)

    # 128x128 리사이즈
    resized = cv2.resize(crop, (INPUT_SIZE, INPUT_SIZE))
    scale_xy = (crop_wh[0] / INPUT_SIZE, crop_wh[1] / INPUT_SIZE)

    # BGR → RGB, HWC → CHW, 정규화 [-1, 1]
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    normalized = (rgb - 0.5) / 0.5
    tensor = torch.from_numpy(normalized.transpose(2, 0, 1)).unsqueeze(0).to(device)

    return tensor, scale_xy, offset_xy


def postprocess_landmarks(output, scale_xy, offset_xy) -> np.ndarray:
    """
    facemap_3dmm 출력 → 원본 이미지 좌표 랜드마크

    Args:
        output   : 모델 출력 (tensor 또는 tuple)
        scale_xy : preprocess에서 반환된 스케일
        offset_xy: preprocess에서 반환된 오프셋
    Returns:
        landmarks: np.ndarray (N, 2) — 원본 이미지 좌표
    """
    # 출력이 tuple인 경우 첫 번째 요소 사용
    if isinstance(output, (tuple, list)):
        lm_tensor = output[0]
    else:
        lm_tensor = output

    lm = lm_tensor.detach().cpu().numpy()

    # shape 정규화 → (N, 2) 또는 (N, 3)
    lm = lm.reshape(-1, lm.shape[-1]) if lm.ndim > 2 else lm
    if lm.shape[-1] >= 2:
        xy = lm[:, :2]  # x, y 만 사용
    else:
        raise ValueError(f'예상치 못한 랜드마크 shape: {lm.shape}')

    # 128x128 좌표 → 원본 이미지 좌표
    sx, sy = scale_xy
    ox, oy = offset_xy
    xy[:, 0] = xy[:, 0] * sx + ox
    xy[:, 1] = xy[:, 1] * sy + oy

    return xy.astype(np.float32)


def landmarks_to_5kps(landmarks_2d: np.ndarray) -> np.ndarray:
    """
    N개 랜드마크 → InsightFace 5-point keypoints
    68점 기준 인덱스 사용. 점 수가 다를 경우 비율로 매핑.
    """
    n = len(landmarks_2d)

    if n == 68:
        right_eye   = landmarks_2d[36:42].mean(axis=0)
        left_eye    = landmarks_2d[42:48].mean(axis=0)
        nose_tip    = landmarks_2d[30]
        right_mouth = landmarks_2d[48]
        left_mouth  = landmarks_2d[54]
    else:
        # 비율 기반 매핑 (68점이 아닌 경우)
        # 랜드마크 수가 너무 적으면 NaN 발생 가능
        if n < 5: # 최소 5개 점은 있어야 유의미한 매핑 가능
            return np.full((5, 2), np.nan, dtype=np.float32)

        def idx(ratio): return int(n * ratio)
        right_eye   = landmarks_2d[idx(0.28):idx(0.38)].mean(axis=0)
        left_eye    = landmarks_2d[idx(0.38):idx(0.48)].mean(axis=0)
        nose_tip    = landmarks_2d[idx(0.44)]
        right_mouth = landmarks_2d[idx(0.60)]
        left_mouth  = landmarks_2d[idx(0.70)]

    return np.array(
        [right_eye, left_eye, nose_tip, right_mouth, left_mouth],
        dtype=np.float32
    )

def is_valid_kps_5(kps: np.ndarray) -> bool:
    """
    5-point keypoints가 유효한지 확인 (shape, NaN 여부)
    """
    if kps is None or kps.shape != (5, 2) or np.any(np.isnan(kps)) or np.any(np.isinf(kps)):
        return False
    return True

def show_images(imgs, titles, figsize=(18, 6)):
    fig, axes = plt.subplots(1, len(imgs), figsize=figsize)
    if len(imgs) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


print('✅ 전처리 / 후처리 / 변환 함수 정의 완료')


# 메인 파이프라인

In [ ]:
def extract_with_facemap(
    img_bgr: np.ndarray,
    insightface_face=None,
    verbose: bool = True
) -> dict:
    """
    facemap_3dmm으로 랜드마크 추출

    Args:
        img_bgr         : BGR 이미지
        insightface_face: InsightFace Face 객체 (bbox 참조용, 없으면 전체 이미지)
    Returns:
        dict with: landmarks_2d, kps_5, detected
    """
    result = dict(landmarks_2d=None, kps_5=None, detected=False)

    # 얼굴 bbox (InsightFace 감지 결과 활용)
    bbox = None
    if insightface_face is not None and hasattr(insightface_face, 'bbox'):
        bbox = insightface_face.bbox

    # facemap_3dmm 전처리
    tensor, scale_xy, offset_xy = preprocess_for_facemap(img_bgr, face_bbox=bbox)

    # 추론
    with torch.no_grad():
        output = facemap_model(tensor)

    # 랜드마크 복원
    landmarks = postprocess_landmarks(output, scale_xy, offset_xy)

    if verbose:
        print(f'  [facemap] 랜드마크 수: {len(landmarks)} | shape: {landmarks.shape}')
        print(f'  [facemap] x범위: {landmarks[:,0].min():.1f}~{landmarks[:,0].max():.1f}')
        print(f'  [facemap] y범위: {landmarks[:,1].min():.1f}~{landmarks[:,1].max():.1f}')

    kps_5 = landmarks_to_5kps(landmarks)

    result.update(dict(landmarks_2d=landmarks, kps_5=kps_5, detected=True))
    # print('FINAL landmarks shape =', np.array(landmarks).shape)
    # print('FINAL landmarks =', landmarks[:10] if np.array(landmarks).ndim == 2 else landmarks)
    # print('FINAL kps_5 =', kps_5)
    return result


def faceswap_pipeline_A(
    source_img_bgr: np.ndarray,
    target_img_bgr: np.ndarray,
    verbose: bool = True
) -> np.ndarray:
    """
    facemap_3dmm → InsightFace inswapper 전체 파이프라인

    Args:
        source_img_bgr: 소스 얼굴 이미지 (BGR)
        target_img_bgr: 타겟 이미지 (이 얼굴에 소스 얼굴을 합성)
    Returns:
        result: BGR numpy array
    """
    # Step 1: InsightFace로 얼굴 감지 (임베딩 추출용)
    src_faces = face_app.get(source_img_bgr)
    if not src_faces:
        raise ValueError('소스 이미지에서 얼굴 감지 실패')
    src_face = src_faces[0]
    if verbose:
        print(f'  [소스] InsightFace 얼굴 {len(src_faces)}개 감지')

    tgt_faces = face_app.get(target_img_bgr)
    if not tgt_faces:
        raise ValueError('타겟 이미지에서 얼굴 감지 실패')
    if verbose:
        print(f'  [타겟] InsightFace 얼굴 {len(tgt_faces)}개 감지')

    # Step 2: facemap_3dmm으로 타겟 랜드마크 추출
    if verbose:
        print('  [facemap_3dmm] 타겟 랜드마크 추출...')
    # NOTE: 현재는 첫 번째 타겟 얼굴에 대해서만 facemap_3dmm을 실행
    #       여러 얼굴에 대해 각각 실행하려면 루프 안에 넣어야 함
    tgt_feat = extract_with_facemap(target_img_bgr, insightface_face=tgt_faces[0], verbose=verbose)

    # Step 3: 얼굴별 swap
    result = target_img_bgr.copy()
    for i, tgt_face in enumerate(tgt_faces):
        # facemap_3dmm 5-kps 주입
        use_facemap_kps = False
        if tgt_feat['detected'] and is_valid_kps_5(tgt_feat['kps_5']):
            # facemap_3dmm 랜드마크가 유효하면 사용
            tgt_face.kps = tgt_feat['kps_5'].astype(np.float32)
            use_facemap_kps = True
        elif hasattr(tgt_face, 'kps') and is_valid_kps_5(tgt_face.kps):
            # facemap_3dmm 랜드마크가 유효하지 않으면 InsightFace 자체 kps 사용
            # tgt_face.kps는 이미 InsightFace가 감지한 kps를 가지고 있음
            pass
        else:
            # 둘 다 유효하지 않으면 해당 얼굴에 대한 스왑 건너뛰기
            if verbose:
                print(f'  [WARN] 얼굴 {i+1}/{len(tgt_faces)}에 대해 유효한 랜드마크를 찾을 수 없어 스왑 건너뜁니다.')
            continue

        if verbose and i == 0:
            if use_facemap_kps:
                print(f'  [kps] facemap_3dmm 기반 5-point kps 주입 완료')
            else:
                print(f'  [kps] facemap_3dmm 랜드마크 유효성 검사 실패. InsightFace 자체 5-point kps 사용.')
            kps_to_log = tgt_face.kps # 현재 tgt_face.kps에 최종 사용될 kps가 들어있음
            kps_labels = ['우측눈', '좌측눈', '코끝', '우측입', '좌측입']
            for label, pt in zip(kps_labels, kps_to_log):
                print(f'         {label}: ({pt[0]:.1f}, {pt[1]:.1f})')

        result = swapper.get(result, tgt_face, src_face, paste_back=True)
        if verbose:
            print(f'  [swap] 얼굴 {i+1}/{len(tgt_faces)} 완료')

    return result


print('✅ 파이프라인 정의 완료')


# 이미지 업로드 및 실행

In [ ]:
import time
import cv2
from google.colab import files

print('📁 소스 이미지 업로드 (얼굴을 가져올 이미지)')
src_upload = files.upload()
src_path = list(src_upload.keys())[0]

print('\n📁 타겟 이미지 업로드 (얼굴을 교체할 이미지)')
tgt_upload = files.upload()
tgt_path = list(tgt_upload.keys())[0]

src_img = cv2.imread(src_path)
tgt_img = cv2.imread(tgt_path)

assert src_img is not None and tgt_img is not None, '이미지 로드 실패'

show_images([src_img, tgt_img], ['소스 (이 얼굴을)', '타겟 (여기에 합성)'])

In [ ]:
import time

print('\n' + '='*55)
print('📍 1단계: 랜드마크 / 얼굴 정보 추출')
print('='*55)

t0 = time.time()

# Step 1: InsightFace로 얼굴 감지
src_faces = face_app.get(src_img)
if not src_faces:
    print('소스 이미지에서 얼굴 감지 실패. 전체 이미지를 사용합니다.')
    src_face_for_facemap = None # Indicate no specific face detected by InsightFace
else:
    src_face_for_facemap = src_faces[0] # Use the first detected face
    print(f'  [소스] InsightFace 얼굴 {len(src_faces)}개 감지')

tgt_faces = face_app.get(tgt_img)
if not tgt_faces:
    print('타겟 이미지에서 얼굴 감지 실패. 전체 이미지를 사용합니다.')
    tgt_face_for_facemap = None # Indicate no specific face detected by InsightFace
else:
    tgt_face_for_facemap = tgt_faces[0] # Use the first detected face
    print(f'  [타겟] InsightFace 얼굴 {len(tgt_faces)}개 감지')


# facemap_3dmm으로 랜드마크 / 얼굴 정보 추출
src_face_info = extract_with_facemap(src_img, insightface_face=src_face_for_facemap, verbose=True)
tgt_face_info = extract_with_facemap(tgt_img, insightface_face=tgt_face_for_facemap, verbose=True)

print(f'\n⏱ 랜드마크/얼굴정보 추출 시간: {time.time()-t0:.2f}초')

# 확인용 출력
print('\n[소스 얼굴 정보]')
for k, v in src_face_info.items():
    if hasattr(v, 'shape'):
        print(f' - {k}: shape={v.shape}')
    else:
        print(f' - {k}: {type(v)}')

print('\n[타겟 얼굴 정보]')
for k, v in tgt_face_info.items():
    if hasattr(v, 'shape'):
        print(f' - {k}: shape={v.shape}')
    else:
        print(f' - {k}: {type(v)}')


def draw_landmarks(image: np.ndarray, landmarks: np.ndarray, color=(0, 255, 0), radius=2) -> np.ndarray:
    """
    이미지에 랜드마크를 그리는 헬퍼 함수
    """
    if landmarks is None or len(landmarks) == 0:
        return image

    vis_image = image.copy()
    for x, y in landmarks:
        if np.isfinite(x) and np.isfinite(y):
            cv2.circle(vis_image, (int(x), int(y)), radius, color, -1)
    return vis_image

print('\n' + '='*55)
print('🖍  랜드마크 시각화')
print('='*55)

src_landmark_vis = draw_landmarks(src_img.copy(), src_face_info['landmarks_2d'])
tgt_landmark_vis = draw_landmarks(tgt_img.copy(), tgt_face_info['landmarks_2d'])

show_images(
    [src_landmark_vis, tgt_landmark_vis],
    ['소스 랜드마크', '타겟 랜드마크']
)

In [ ]:
def apply_faceswap_A(
    source_img_bgr: np.ndarray,
    target_img_bgr: np.ndarray,
    src_faces: list, # List of InsightFace Face objects from face_app.get()
    tgt_faces: list, # List of InsightFace Face objects from face_app.get()
    src_face_info: dict, # Dict from extract_with_facemap for source
    tgt_face_info: dict, # Dict from extract_with_facemap for target
    verbose: bool = True
) -> np.ndarray:
    """
    사전에 추출된 얼굴 정보(InsightFace Face 객체 및 facemap_3dmm 랜드마크)를 사용하여
    InsightFace inswapper로 얼굴 스왑을 적용하는 함수.
    """
    if not src_faces:
        raise ValueError('소스 이미지에서 얼굴 감지 실패')
    src_face = src_faces[0] # 첫 번째 소스 얼굴을 스왑 소스로 사용

    result = target_img_bgr.copy()
    for i, tgt_face in enumerate(tgt_faces):
        # facemap_3dmm 5-kps 주입
        use_facemap_kps = False
        if i == 0 and tgt_face_info['detected'] and is_valid_kps_5(tgt_face_info['kps_5']):
            # 첫 번째 타겟 얼굴에만 facemap_3dmm 랜드마크를 적용
            tgt_face.kps = tgt_face_info['kps_5'].astype(np.float32)
            use_facemap_kps = True
        elif hasattr(tgt_face, 'kps') and is_valid_kps_5(tgt_face.kps):
            # facemap_3dmm 랜드마크가 유효하지 않으면 InsightFace 자체 kps 사용
            pass
        else:
            # 둘 다 유효하지 않으면 해당 얼굴에 대한 스왑 건너뛰기
            if verbose:
                print(f'  [WARN] 얼굴 {i+1}/{len(tgt_faces)}에 대해 유효한 랜드마크를 찾을 수 없어 스왑 건너뜁니다.')
            continue

        if verbose and i == 0:
            if use_facemap_kps:
                print(f'  [kps] facemap_3dmm 기반 5-point kps 주입 완료')
            else:
                print(f'  [kps] facemap_3dmm 랜드마크 유효성 검사 실패. InsightFace 자체 5-point kps 사용.')
            kps_to_log = tgt_face.kps # 현재 tgt_face.kps에 최종 사용될 kps가 들어있음
            kps_labels = ['우측눈', '좌측눈', '코끝', '우측입', '좌측입']
            for label, pt in zip(kps_labels, kps_to_log):
                print(f'         {label}: ({pt[0]:.1f}, {pt[1]:.1f})')

        result = swapper.get(result, tgt_face, src_face, paste_back=True)
        if verbose:
            print(f'  [swap] 얼굴 {i+1}/{len(tgt_faces)} 완료')

    return result


print('\n' + '='*55)
print('🎭 2단계: 모델 적용 얼굴 스왑')
print('='*55)

t0 = time.time()

result_img = apply_faceswap_A(
    source_img_bgr=src_img,
    target_img_bgr=tgt_img,
    src_faces=src_faces,
    tgt_faces=tgt_faces,
    src_face_info=src_face_info,
    tgt_face_info=tgt_face_info,
    verbose=True
)

print(f'\n⏱ 얼굴 스왑 시간: {time.time()-t0:.2f}초')
print('='*55)

cv2.imwrite('/content/result_A.jpg', result_img)
show_images([src_img, tgt_img, result_img], ['소스', '타겟 원본', '✅ Face Swap 결과'])
print('💾 저장: /content/result_A.jpg')
